# Ch 14. アテンション機構 — 解答

> このノートブックは5問すべての厳密な解答と再現可能な検証を含む。

## 問題 1

**問題**: $\mathrm{Var}(QK^\top_{ij}) = d_k$ Q, K iid $\mathcal{N}(0,1)$ 複雑度.

### 厳密な解説

制御する要因: **key dimension d_k**。  測定指標: **empirical variance of Q dot K**。  解釈と限界: Each product Q_l K_l has mean zero and variance one; independence makes variances add, giving d_k. Monte Carlo error is reported instead of asserted as an exact sample identity.


In [ ]:
import torch
torch.manual_seed(140)
for d in (8,32,128):
 q=torch.randn(30000,d); k=torch.randn(30000,d); empirical=(q*k).sum(1).var().item(); print({'d_k':d,'empirical_variance':empirical,'relative_error':abs(empirical-d)/d})


## 問題 2

**問題**: Attention Attention 出力 比較.

### 厳密な解説

制御する要因: **mask: none versus causal**。  測定指標: **maximum attention mass assigned to future positions and output difference**。  解釈と限界: The mask sets future logits to -infinity before softmax, so causal future mass is exactly zero. Q, K, and V remain identical.


In [ ]:
import torch
torch.manual_seed(141); q=torch.randn(1,1,5,8); k=torch.randn(1,1,5,8); v=torch.randn(1,1,5,8); score=q@k.transpose(-2,-1)/8**.5; future=torch.triu(torch.ones(5,5,dtype=torch.bool),1); a=score.softmax(-1); ac=score.masked_fill(future,float('-inf')).softmax(-1); plain=a@v; causal=ac@v; print({'plain_future_mass':a.masked_select(future).max().item(),'causal_future_mass':ac.masked_select(future).max().item(),'output_l2_difference':(plain-causal).norm().item()}); assert ac.masked_select(future).max()==0


## 問題 3

**問題**: Attention 行列 可視化, .

### 厳密な解説

制御する要因: **query/key construction: aligned positional features versus shuffled keys**。  測定指標: **attention matrix and mean diagonal weight**。  解釈と限界: When each query equals its same-position key, self dot products exceed cross-position dot products. Large diagonals are therefore data/model dependent, not guaranteed by attention itself.


In [ ]:
import torch
q=torch.eye(6)*3; k=q.clone(); scores=q@k.T/6**.5; a=scores.softmax(-1); shuffled=a[:,torch.tensor([1,2,3,4,5,0])]; print({'aligned_attention':a.round(decimals=3).tolist(),'aligned_diagonal_mean':a.diag().mean().item(),'shuffled_diagonal_mean':shuffled.diag().mean().item()}); assert a.diag().mean()>shuffled.diag().mean()


## 問題 4

**問題**: シーケンス長 256, 512, 1024, 2048 CPU 時間 $O(n^2)$ 検証.

### 厳密な解説

制御する要因: **sequence length: 256, 512, 1024, 2048**。  測定指標: **median CPU time and normalized time/n^2**。  解釈と限界: Single-threaded matrix multiplication is timed after warmup. Runtime includes kernel effects, so n^2 element count is also printed as the exact complexity metric.


In [ ]:
import torch
import time, statistics
torch.set_num_threads(1); torch.manual_seed(142); d=16
for n in (256,512,1024,2048):
 q=torch.randn(n,d); k=torch.randn(n,d); _=q@k.T; times=[]
 for _ in range(3): t=time.perf_counter(); _=q@k.T; times.append(time.perf_counter()-t)
 med=statistics.median(times); print({'n':n,'median_seconds':med,'score_elements':n*n,'seconds_per_n2':med/(n*n)})


## 問題 5

**問題**: PyTorch SDPA `is_causal=True` Attention .

### 厳密な解説

制御する要因: **implementation: PyTorch SDPA is_causal=True versus explicit causal mask**。  測定指標: **maximum absolute output error**。  解釈と限界: The explicit reference applies the same scale, triangular mask, softmax, and value product. Agreement verifies the SDPA causal option on CPU.


In [ ]:
import torch
torch.manual_seed(143); q=torch.randn(2,3,5,8); k=torch.randn(2,3,5,8); v=torch.randn(2,3,5,8); got=torch.nn.functional.scaled_dot_product_attention(q,k,v,is_causal=True); mask=torch.triu(torch.ones(5,5,dtype=torch.bool),1); score=(q@k.transpose(-2,-1))/8**.5; ref=score.masked_fill(mask,float('-inf')).softmax(-1)@v; err=(got-ref).abs().max().item(); print({'shape':list(got.shape),'max_reference_error':err}); assert err<1e-5
